<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/03-dataframe-fundamentals/01-select-filter-withcolumn.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 3.1 — select, filter/where, withColumn, drop, alias, cast

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

spark = (
    SparkSession.builder
    .appName("3.1-core-verbs")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ORDERS_SCHEMA = StructType([
    StructField("order_id",    IntegerType(), False),
    StructField("customer_id", StringType(),  False),
    StructField("order_date",  DateType(),    True),
    StructField("product",     StringType(),  True),
    StructField("category",    StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(),  True),
    StructField("country",     StringType(),  True),
])
orders = (
    spark.read.option("header", True)
    .schema(ORDERS_SCHEMA)
    .csv(f"{DATA_DIR}/orders.csv")
)
orders.show(3)

## select: projection + computed columns (alias everything computed)

In [ ]:
orders.select(
    "order_id",
    "product",
    (col("quantity") * col("unit_price")).alias("line_total"),
).show(5)

# without alias — look at the generated column name:
orders.select(col("quantity") * col("unit_price")).columns

## filter: & | ~ and mandatory parentheses

In [ ]:
orders.where((col("country") == "DE") & (col("category") == "books")).show()

# The classic error, on purpose — read the message you'll see for years:
try:
    orders.filter(col("country") == "IN" and col("quantity") > 1)
except Exception as e:
    print(type(e).__name__, "->", str(e).splitlines()[0])

## withColumn: add, replace — and the no-op bug

In [ ]:
orders2 = orders.withColumn("line_total", col("quantity") * col("unit_price"))
print("orders2 columns:", "line_total" in orders2.columns)

# Immutability: the original is untouched — the assignment IS the operation
orders.withColumn("oops", col("order_id") + 1)   # result discarded!
print("orders columns: ", "oops" in orders.columns)

## cast: impossible casts become null, silently

In [ ]:
messy = spark.createDataFrame(
    [("12.99",), ("N/A",), ("8.50",)], schema="price STRING"
)
casted = messy.withColumn("price_d", col("price").cast("double"))
casted.show()
print("nulls created:", casted.filter(col("price_d").isNull()).count())

## drop, rename — and the chained-pipeline idiom

In [ ]:
print(orders.drop("produt").columns == orders.columns)   # typo dropped NOTHING, silently

result = (
    orders
    .filter(col("category") != "furniture")
    .withColumn("line_total", col("quantity") * col("unit_price"))
    .select("order_id", "customer_id", "country", "line_total")
    .filter(col("line_total") > 20)
    .withColumnRenamed("country", "country_code")
)
result.show(5)

## Exercises

1. Electronics orders from IN or UK with `line_total` > 25, showing order_id, product, line_total — one chain.
2. Add `order_month` by casting/formatting `order_date` (try `col("order_date").cast("string").substr(1, 7)` — a preview of string functions).
3. Demonstrate the filter-null teaser from the lesson: count rows returned by `filter(col("country") == "IN")` plus rows from `filter(col("country") != "IN")`. Do they add up to 41? Where did the missing rows go? (Full story: lesson 3.5.)

In [ ]:
# your exercise code here
